# TPE Optimizer (Optuna)

Joint-portfolio TPE search. See `docs/research.md` for methodology.

## Setup

In [6]:
# ruff: noqa: E402
import sys
from pathlib import Path

for _candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (_candidate / "prediction_market_extensions").is_dir() and (
        _candidate / "backtests"
    ).is_dir():
        if str(_candidate) not in sys.path:
            sys.path.insert(0, str(_candidate))
        break

from prediction_market_extensions.backtesting._notebook_support import ensure_notebook_repo_context

repo_root = ensure_notebook_repo_context()

## Configuration

In [7]:
from decimal import Decimal

from prediction_market_extensions.backtesting._execution_config import ExecutionModelConfig
from prediction_market_extensions.backtesting._execution_config import StaticLatencyConfig
from prediction_market_extensions.backtesting._experiments import ParameterSearchExperiment
from prediction_market_extensions.backtesting._prediction_market_runner import MarketDataConfig
from prediction_market_extensions.backtesting._replay_specs import QuoteReplay
from prediction_market_extensions.backtesting.data_sources import PMXT, Polymarket, QuoteTick
from prediction_market_extensions.backtesting.optimizers import ParameterSearchConfig
from prediction_market_extensions.backtesting.optimizers import ParameterSearchWindow


NAME = "notebook_tpe_optimizer"

DATA = MarketDataConfig(
    platform=Polymarket,
    data_type=QuoteTick,
    vendor=PMXT,
    sources=(
        "local:/Volumes/LaCie/pmxt_raws",
        "archive:r2.pmxt.dev",
        "relay:209-209-10-83.sslip.io",
    ),
)

# Joint-portfolio: every trial evaluates ONE parameter set across all 5 replays
# simultaneously. PnL and fills sum, drawdown is computed on the summed equity
# curve, coverage is averaged across markets.
#
# IMPORTANT: ParameterSearchWindow applies a uniform time range to every replay,
# so BASE_REPLAYS must only contain slugs whose active quote-data window covers
# the TRAIN/HOLDOUT ranges below. These five are taken from
# polymarket_quote_tick_joint_portfolio_runner.py and all have data across
# 2026-04-06 12:00Z → 2026-04-07 18:00Z. If you swap in a market with a
# narrower window, tighten the train/holdout windows to match or the optimizer
# will reject every trial with invalid_result_count.
BASE_REPLAYS = (
    QuoteReplay(market_slug="will-ludvig-aberg-win-the-2026-masters-tournament", token_index=0),
    QuoteReplay(
        market_slug="will-the-tennessee-titans-draft-a-quarterback-in-the-first-round-of-the-2026-nfl-draft",
        token_index=0,
    ),
    QuoteReplay(
        market_slug="will-the-south-african-reserve-bank-decrease-the-repo-rate-after-the-may-meeting",
        token_index=0,
    ),
    QuoteReplay(market_slug="will-nana-araba-wilmot-win-top-chef-season-23", token_index=0),
    QuoteReplay(market_slug="will-drake-release-an-album-in-2026", token_index=0),
)

TRAIN_WINDOWS = (
    ParameterSearchWindow(
        name="sample-a-wide",
        start_time="2026-04-06T12:00:00Z",
        end_time="2026-04-07T18:00:00Z",
    ),
    ParameterSearchWindow(
        name="sample-b-overnight",
        start_time="2026-04-06T18:00:00Z",
        end_time="2026-04-07T12:00:00Z",
    ),
    ParameterSearchWindow(
        name="sample-c-morning",
        start_time="2026-04-07T00:00:00Z",
        end_time="2026-04-07T12:00:00Z",
    ),
)

HOLDOUT_WINDOWS = (
    ParameterSearchWindow(
        name="sample-d-afternoon",
        start_time="2026-04-07T12:00:00Z",
        end_time="2026-04-07T18:00:00Z",
    ),
)

STRATEGY_SPEC = {
    "strategy_path": "strategies:QuoteTickEMACrossoverStrategy",
    "config_path": "strategies:QuoteTickEMACrossoverConfig",
    "config": {
        "trade_size": Decimal("5"),
        "fast_period": "__SEARCH__:fast_period",
        "slow_period": "__SEARCH__:slow_period",
        "entry_buffer": "__SEARCH__:entry_buffer",
        "take_profit": "__SEARCH__:take_profit",
        "stop_loss": "__SEARCH__:stop_loss",
    },
}

# Continuous / log-scale space — TPE samples within these bounds.
PARAMETER_SPACE = {
    "fast_period": {"type": "int", "low": 16, "high": 128},
    "slow_period": {"type": "int", "low": 96, "high": 512},
    "entry_buffer": {"type": "float", "low": 1e-4, "high": 2e-3, "log": True},
    "take_profit": {"type": "float", "low": 2e-3, "high": 3e-2, "log": True},
    "stop_loss": {"type": "float", "low": 2e-3, "high": 3e-2, "log": True},
}

EXECUTION = ExecutionModelConfig(
    queue_position=True,
    latency_model=StaticLatencyConfig(
        base_latency_ms=75.0,
        insert_latency_ms=10.0,
        update_latency_ms=5.0,
        cancel_latency_ms=5.0,
    ),
)

MAX_TRIALS = 50
HOLDOUT_TOP_K = 5

PARAMETER_SEARCH = ParameterSearchConfig(
    name=NAME,
    data=DATA,
    base_replays=BASE_REPLAYS,
    strategy_spec=STRATEGY_SPEC,
    parameter_space=PARAMETER_SPACE,
    train_windows=TRAIN_WINDOWS,
    holdout_windows=HOLDOUT_WINDOWS,
    max_trials=MAX_TRIALS,
    random_seed=7,
    holdout_top_k=HOLDOUT_TOP_K,
    initial_cash=100.0,
    probability_window=256,
    min_quotes=500,
    min_price_range=0.005,
    min_fills_per_window=1,
    execution=EXECUTION,
    emit_html=False,
    chart_output_path="output",
    sampler="tpe",
)

EXPERIMENT = ParameterSearchExperiment(
    name=NAME,
    description="EMA TPE joint-portfolio search on PMXT L2 data",
    parameter_search=PARAMETER_SEARCH,
)

print(
    f"Configured TPE search: {MAX_TRIALS} trials over "
    f"{len(BASE_REPLAYS)} markets (joint portfolio), "
    f"{len(TRAIN_WINDOWS)} train window(s), {len(HOLDOUT_WINDOWS)} holdout window(s)."
)

Configured TPE search: 50 trials over 5 markets (joint portfolio), 3 train window(s), 1 holdout window(s).


## Run optimizer

In [8]:
from pprint import pprint

from prediction_market_extensions.backtesting._experiments import run_experiment_async
from prediction_market_extensions.backtesting._notebook_support import suppress_notebook_cell_output

with suppress_notebook_cell_output():
    summary = await run_experiment_async(EXPERIMENT)

print("Selected params:")
pprint(dict(summary.selected_params))
print("Leaderboard CSV:", summary.leaderboard_csv_path)
print("Summary JSON:", summary.summary_json_path)

Selected params:
{'entry_buffer': 0.0003718635067633045,
 'fast_period': 24,
 'slow_period': 421,
 'stop_loss': 0.02826408380687946,
 'take_profit': 0.014187016294426116}
Leaderboard CSV: /Users/evankolberg/prediction-market-backtesting/output/notebook_tpe_optimizer_leaderboard.csv
Summary JSON: /Users/evankolberg/prediction-market-backtesting/output/notebook_tpe_optimizer_summary.json


## Leaderboard

In [ ]:
import pandas as pd
from IPython.display import display, Markdown

_leaderboard_df = pd.read_csv(summary.leaderboard_csv_path)
_display_cols = [
    c
    for c in (
        "trial_id",
        "train_median_score",
        "holdout_median_score",
        "train_median_pnl",
        "holdout_median_pnl",
        "train_median_drawdown",
        "train_median_fills",
        "params_json",
    )
    if c in _leaderboard_df.columns
]
_top = _leaderboard_df[_display_cols].head(10).copy()

_fmt = {
    "train_median_score": "{:.4f}".format,
    "holdout_median_score": "{:.4f}".format,
    "train_median_pnl": "{:.4f}".format,
    "holdout_median_pnl": "{:.4f}".format,
    "train_median_drawdown": "{:.4f}".format,
    "train_median_fills": "{:.1f}".format,
}
_fmt = {k: v for k, v in _fmt.items() if k in _top.columns}

_styled = (
    _top.style.format(_fmt, na_rep="-")
    .hide(axis="index")
    .set_caption(f"Top candidates — {EXPERIMENT.name} (TPE)")
)
display(_styled)

display(
    Markdown(
        "### Selected parameters\n\n"
        + "\n".join(f"- **{k}**: `{v}`" for k, v in dict(summary.selected_params).items())
    )
)

trial_id,train_median_score,holdout_median_score,train_median_pnl,holdout_median_pnl,train_median_drawdown,train_median_fills,params_json
1,-99.7450,-1000000000.0000,-99.7450,0.0000,0.0000,348.0,"{""entry_buffer"": 0.0003718635067633045, ""fast_period"": 24, ""slow_period"": 421, ""stop_loss"": 0.02826408380687946, ""take_profit"": 0.014187016294426116}"
40,-99.7750,-1000000000.0000,-99.7750,0.0000,0.0000,340.0,"{""entry_buffer"": 0.0007792909159842779, ""fast_period"": 112, ""slow_period"": 438, ""stop_loss"": 0.016305555044155075, ""take_profit"": 0.011741515415871732}"
3,-99.7950,-1000000000.0000,-99.7950,0.0000,0.0000,340.0,"{""entry_buffer"": 0.00031305153729655524, ""fast_period"": 92, ""slow_period"": 431, ""stop_loss"": 0.004364309247234048, ""take_profit"": 0.00239098668116823}"
43,-99.7950,-1000000000.0000,-99.7950,0.0000,0.0000,340.0,"{""entry_buffer"": 0.0009525701632838303, ""fast_period"": 80, ""slow_period"": 487, ""stop_loss"": 0.028747690927671692, ""take_profit"": 0.015100495372397502}"
21,-99.8100,-1000000000.0000,-99.8100,0.0000,0.0000,340.0,"{""entry_buffer"": 0.0005098673948717401, ""fast_period"": 49, ""slow_period"": 455, ""stop_loss"": 0.0030120456692229736, ""take_profit"": 0.006595593297112331}"
28,-99.8100,-,-99.8100,-,0.0000,340.0,"{""entry_buffer"": 0.0005491907610335264, ""fast_period"": 78, ""slow_period"": 401, ""stop_loss"": 0.0035647981370876074, ""take_profit"": 0.017120609544050636}"
35,-99.8100,-,-99.8100,-,0.0000,340.0,"{""entry_buffer"": 0.0005899666898412243, ""fast_period"": 77, ""slow_period"": 393, ""stop_loss"": 0.004385949882797099, ""take_profit"": 0.016250761055832088}"
6,-99.8250,-,-99.8250,-,0.0000,346.0,"{""entry_buffer"": 0.0009469035303373606, ""fast_period"": 31, ""slow_period"": 314, ""stop_loss"": 0.0070982296398728, ""take_profit"": 0.012241950538095943}"
11,-99.8293,-,-99.8293,-,0.0000,302.0,"{""entry_buffer"": 0.0006511189611833273, ""fast_period"": 52, ""slow_period"": 114, ""stop_loss"": 0.016196138706714668, ""take_profit"": 0.01844145737364155}"
38,-99.8471,-,-99.8471,-,0.0000,354.0,"{""entry_buffer"": 0.0005348991334094322, ""fast_period"": 36, ""slow_period"": 195, ""stop_loss"": 0.004053142417411394, ""take_profit"": 0.019716272791856698}"


### Selected parameters

- **fast_period**: `24`
- **slow_period**: `421`
- **entry_buffer**: `0.0003718635067633045`
- **take_profit**: `0.014187016294426116`
- **stop_loss**: `0.02826408380687946`

: 

## Holdout replay

In [ ]:
import gc

from prediction_market_extensions.backtesting._notebook_support import (
    display_html_artifacts,
    find_updated_html_artifacts,
    snapshot_html_artifacts,
    suppress_notebook_cell_output,
)
from prediction_market_extensions.backtesting.optimizers import (
    build_parameter_search_window_backtest,
)

selected_window = HOLDOUT_WINDOWS[0] if HOLDOUT_WINDOWS else TRAIN_WINDOWS[-1]

output_root = repo_root / "output"
html_snapshot = snapshot_html_artifacts(output_root)

backtest = build_parameter_search_window_backtest(
    config=PARAMETER_SEARCH,
    window=selected_window,
    params=summary.selected_params,
    trial_id=0,
    name=f"{NAME}:{selected_window.name}:selected-params",
    emit_html=True,
    chart_output_path=str(output_root),
    return_summary_series=False,
)

with suppress_notebook_cell_output():
    results = await backtest.run_async()

updated_html = find_updated_html_artifacts(output_root, html_snapshot)
num_markets = len(results)
print(f"Replayed {num_markets} market(s) on window '{selected_window.name}'.")
display_html_artifacts(updated_html, repo_root=repo_root)

# Holdout replay runs in-kernel with full chart emission — drop refs and GC
# to stop memory accumulating across re-runs. Without this the kernel grows
# ~500 MB per re-run and macOS SIGKILLs it after a few iterations.
del results, backtest, updated_html, html_snapshot
gc.collect()